# Publisher Application

This application makes use of the Confluence Publisher class defined in this folder. See ./Publisher.py

## Load configuration
Be aware not to commit your credentials!

In [ ]:
import yaml
import copy
import logging
log = logging.getLogger(__name__)

with open('config.yaml') as f:
    config = yaml.safe_load(f)

conf_conf = config['confluence']
assert conf_conf
assert len(conf_conf['apiurl']) > 0
space_key = conf_conf['space']
root_page = conf_conf['rootpage']

conf_confidential = copy.deepcopy(config)
conf_confidential['confluence']['password'] = '***'
conf_confidential

In [ ]:
# enable overwrite by test infrastructure
confluence_username = config['confluence']['username']
confluence_password = config['confluence']['password']

## Load the data
The **data** is the JSON serialized information model 

In [ ]:
import json

data = None
with open(config['json'], 'r') as source:
     data = json.load(source)

print('Model "{}" contains {} entities and {} attibutes'.format(
    data['model']['name'], len(data['entities']), len(data['attributes'])))

The [Confluence API](https://github.com/atlassian-api/atlassian-python-api) is embedded as a **git submodule** in the 'lib' folder next to this notebook.

Use `git submodule update --init` to fetch all submodules after a checkout without `--recursive` option.

If the next cell fails, install confluence-api submodule from the repository root with:
`git submodule add -f https://github.com/atlassian-api/atlassian-python-api.git notebooks/contentfactory/lib/atlassian-python-api`

In [ ]:
import sys
import os

library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence

In [ ]:
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])
root_page_id

In [ ]:
import Publisher as cp
publisher = cp.Publisher(config, data, confluence, space_key, root_page_id, language=config['language'])
list(map(lambda e: (e, publisher.translate(data['entities'][e]['name'])), list(data['entities'])[:3]))

In [ ]:
publisher.scan_current_content()

# Prepare destination structure

In [ ]:
import ipywidgets
from IPython.display import display

for element_class in config['content']:
    entry_config = config['content'][element_class]
    element_title = element_class
    if entry_config.get('title'):
        element_title = entry_config['title']
    folder_created = publisher.stub(element_title, root_page_id)
    folder_page_id = folder_created['id']
    print('Creating group page "{}". Confluence page id:{}'.format(element_title, str(folder_page_id)))
    
    progress_bar = ipywidgets.IntProgress(min=0, max=len(data[element_class]), 
        description='Processing content class "{}" ({})'.format(element_class, len(data[element_class])),
        layout={'width': '100%'}, style = {'description_width':'initial'})
    display(progress_bar)
    
    for item in data[element_class]:
        progress_bar.value += 1
        entry_data = data[element_class][item]
       
        title = publisher.translate(entry_data['name'])
        safe_title = title.strip()
        if safe_title != title:
            log.warning('{} title "{}" violates title constraints. Corrected to "{}"'.format(element_class, title, safe_title))

        result = publisher.stub(safe_title, folder_page_id)
        page_id = result['id']
        publisher.register_page(item, page_id, safe_title)
        for label in entry_config['labels']:
            confluence.set_page_label(page_id, 'test-delete-freely')

    progress_bar.bar_style = "success"


## Now generate content

In [ ]:
publisher.content_map

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape
import re

jinja_env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml'])
)

for element_class in config['content']:
    print('Creating content for class "{}" ({})'.format(element_class, len(data[element_class])))
    entry_config = config['content'][element_class]
    
    progress_bar = ipywidgets.IntProgress(min=0, max=len(data[element_class]), 
        description='Processing content class "{}" ({})'.format(element_class, len(data[element_class])),
        layout={'width': '100%'}, style = {'description_width':'initial'})
    display(progress_bar)
    
    jinja_template = jinja_env.get_template(entry_config['template'])
    
    for item in data[element_class]:
        progress_bar.value += 1
        item_data = data[element_class][item]
        try:
            rendered = jinja_template.render(data=data, key=item, item=item_data, util=publisher)
            content_xml = re.sub('<!--.+?->(\n+)*', '', rendered) # strip comment lines
            publisher.update_page(item, content_xml)
        except Exception as e:
            log.error('Unable to process {} {}'.format(element_class, item), e)
        
    progress_bar.bar_style = "success"